# Gold - Lakehouse Medalhão de RH (Databricks)

Continuação de `silver.ipynb` desta mesma pasta: lê as tabelas já publicadas em `{catalogo}.silver` e monta o esquema estrela em `{catalogo}.gold`, espelhando `etl_gold_estrutura_organizacional` e `etl_gold_fato_eventos_rh` do notebook `demonstracao-lakehouse-medalhao.ipynb`.

> ⚠️ **Pré-requisito:** o schema `gold` já deve existir no catálogo informado abaixo (`CREATE SCHEMA IF NOT EXISTS <catalogo>.gold`).

In [ ]:
dbutils.widgets.text('catalogo', 'meu_catalogo')
catalogo = dbutils.widgets.get('catalogo')

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## `estrutura_organizacional`

Conta quantos departamentos subordinados cada departamento tem — a mesma agregação simples de `etl_gold_estrutura_organizacional`, agora com `groupBy`/`agg` no lugar de `groupby`/`size` do pandas.

In [ ]:
def gold_estrutura_organizacional():
    dim_departamento = spark.table(f'{catalogo}.silver.departamentos')

    # conta quantos departamentos têm cada departamento_id como pai
    df_gold = (
        dim_departamento
        .filter(F.col('departamento_pai_id').isNotNull())
        .groupBy('departamento_pai_id')
        .agg(F.count('*').alias('qtd_subdepartamentos'))
        .withColumnRenamed('departamento_pai_id', 'departamento_id')
        .join(dim_departamento.select('departamento_id', 'nome_departamento'), on='departamento_id', how='left')
        .orderBy(F.col('qtd_subdepartamentos').desc())
    )

    df_gold.write.format('delta').mode('overwrite').saveAsTable(f'{catalogo}.gold.estrutura_organizacional')
    print(f'[GOLD] "estrutura_organizacional" gerada com {df_gold.count()} linhas.')
    return df_gold


df_estrutura_organizacional = gold_estrutura_organizacional()

## `fato_eventos_rh`: o Join Temporal

Equivalente Spark de `etl_gold_fato_eventos_rh` (pandas): liga cada evento à versão de `dim_funcionario` vigente na sua data (`data_evento` dentro de `[data_inicio_vigencia, data_fim_vigencia]`), tratando a primeira versão conhecida de cada funcionário como válida também para eventos anteriores ao início da captura (a mesma limitação de SCD Tipo 2 discutida na demonstração).

In [ ]:
def gold_fato_eventos_rh():
    df_eventos = spark.table(f'{catalogo}.silver.eventos')
    dim_funcionario = spark.table(f'{catalogo}.silver.dim_funcionario')

    # a primeira versão de cada funcionário também cobre eventos anteriores ao início da captura
    janela_funcionario = Window.partitionBy('funcionario_id')
    dim_funcionario = (
        dim_funcionario
        .withColumn('_primeira_versao_data', F.min('data_inicio_vigencia').over(janela_funcionario))
        .withColumn('primeira_versao', F.col('data_inicio_vigencia') == F.col('_primeira_versao_data'))
        .drop('_primeira_versao_data')
    )

    # join temporal: liga cada evento à versão do funcionário vigente na data do evento
    fato = df_eventos.join(
        dim_funcionario.select('sk_funcionario', 'funcionario_id', 'data_inicio_vigencia', 'data_fim_vigencia', 'primeira_versao'),
        on='funcionario_id',
        how='left',
    )

    vigente_na_data = (
        (F.col('data_evento') >= F.col('data_inicio_vigencia'))
        | (F.col('primeira_versao') & (F.col('data_evento') < F.col('data_inicio_vigencia')))
    ) & (
        F.col('data_fim_vigencia').isNull() | (F.col('data_evento') <= F.col('data_fim_vigencia'))
    )

    df_gold = (
        fato
        .filter(vigente_na_data)
        .select(
            'evento_id', 'sk_funcionario', 'funcionario_id', 'data_evento', 'tipo_evento',
            'departamento_id', 'cargo_id', 'salario_anterior', 'salario_novo', 'motivo',
        )
        .orderBy('data_evento')
    )

    df_gold.write.format('delta').mode('overwrite').saveAsTable(f'{catalogo}.gold.fato_eventos_rh')
    print(f'[GOLD] tabela fato "fato_eventos_rh" gerada com {df_gold.count()} linhas (grão: 1 linha = 1 evento de RH).')
    return df_gold


df_fato_eventos_rh = gold_fato_eventos_rh()

Com `estrutura_organizacional` e `fato_eventos_rh` publicadas em `{catalogo}.gold`, o esquema estrela está pronto para consumo — os mesmos joins/agregações vistos na seção "Consultando o Esquema Estrela" da demonstração (por exemplo, promoções por departamento) funcionam aqui trocando `pd.read_csv` por `spark.table(f'{catalogo}.gold....')`.